# K4TQ-A "Cathy" — the wing section: NeuralFoil, then a CFD check

Which airfoil for the cruise, the speed, the hover and the transition — losses against lift, phase by phase, for the
**front** and the **rear** wing (in a tandem they fly at different lift coefficients: the CG sets the split).

1. **NeuralFoil** (`vegeta.zephyr`, the surrogate Vegeta imports): 14 candidates at each wing's Reynolds numbers, cached
   in `designs/data/k4tq/sections_neuralfoil.json` (**CALC**). Seconds.
2. **CFD** (`vegeta.aeromant`, OpenFOAM steady RANS k-ω SST): a slab of each leader at a few angles on a *quick* mesh, to
   confirm NeuralFoil's ranking — not its absolute numbers. Mocked while `DEBUG = True` (**MOCK**, pipeline only).
   With `DEBUG = False` and OpenFOAM installed (`./install_local.sh`) it runs for real (**SOLVER**): ~3–6 min per case
   on 4 cores at `QUALITY = "quick"`, the default 8 cases ≈ 30–45 min.

Module: `designs/k4tq_sections.py`. Report: `designs/data/k4tq/sections.md`.

In [ ]:
DEBUG = True            # True: the CFD is mocked (seconds, MOCK). False: OpenFOAM runs for real (SOLVER)
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display
import k4tq, k4tq_sizing as sz, k4tq_sections as ks
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
veh = sz.vehicle(k4tq.MAIN)
print(veh["p"]["variant"], k4tq.NICKNAME.get(veh["p"]["variant"], ""), f"{veh['mass']['mass_kg']:.2f} kg")

## 1. NeuralFoil — every candidate, both wings

In [ ]:
res = ks.study(veh)
cond = res["conditions"]
pd.DataFrame({w: {f"{ph} {q}": (round(v[q], 3) if q == "cl" else round(v[q], -3) if q == "Re" else round(v[q], 1))
                  for ph, v in cond[w].items() if isinstance(v, dict) for q in v} for w in cond})

In [ ]:
cols = ["cruise cd", "cruise L/D", "speed cd", "hover cd (α 0)", "hover cl (α 0)", "transition cl_max", "lift kept 4° past the stall",
        "cruise cm", "t/c at 30 %", "spar depth on the tip [mm]", "least confidence", "score cruise", "score speed", "score hover", "score transition", "total"]
for w in ("front", "rear"):
    display(Markdown(f"### {w} wing — recommendation: **{res['recommendation'][w]}**"))
    display(res["tables"][w][cols].astype(float).round(4))

In [ ]:
fig = ks.figure(res, ks.DATA / "figures" / "k4tq_sections.png")
Image(str(fig))

**Reading it.** Cruise is 60 % of the weight (the energy), hover and transition 15 % each, speed 10 %. A section whose
spar depth at 30 % of the tip chord is under 18 mm (Ø16 spar + skins) scores 0 — on the front wing (120 mm tip chord)
that leaves only the 15 % sections; a tapered spar (Ø12 at the tip) would let the thinner, slipperier ones back in.
Symmetric sections (0012 / 0015) pull nothing sideways in the hover slipstream but pay in cruise drag at the front
wing's high cl; the cambered 4415 wins the front wing on cruise L/D and on a gentle, high-cl stall.

## 2. CFD check of the leaders (Aeromant, a slab of the section)

In [ ]:
QUALITY = "quick"                     # "quick" (default, coarse) or "screening" (finer, ~2× the time)
ALPHAS = (4.0, 10.0)                   # the cruise-like angle and one near the stall; add 0, 2, 6 ... for a polar
LEADERS = {"front": list(res["tables"]["front"].index[:2]), "rear": list(res["tables"]["rear"].index[:2])}
RUNS = Path(ks.DATA / "cfd_sections")
specs = {}
for w, names in LEADERS.items():
    specs.update({f"{w}: {k}": v for k, v in ks.cfd_specs(names, cond, alphas=ALPHAS, which=w).items()})
pd.DataFrame(specs).T

In [ ]:
from vegeta import aeromant
cases = {k: ks.cfd_case(RUNS, v, quality=QUALITY) for k, v in specs.items()}
results = dict(zip(cases, aeromant.run_cases(list(cases.values()), jobs=1, processors=max(1, (os.cpu_count() or 2) // 2), progress=True)))
table = ks.cfd_table(results, specs)
table

In [ ]:
import matplotlib.pyplot as plt
ok = table[table["status"].isin(["SOLVER", "MOCK"])]
fig, ax = plt.subplots(figsize=(7, 4))
if len(ok):
    ax.scatter(ok["NeuralFoil cl"].astype(float), ok["CFD cl (section est.)"].astype(float))
    for k, r in ok.iterrows():
        ax.annotate(k.split(": ")[1], (float(r["NeuralFoil cl"]), float(r["CFD cl (section est.)"])), fontsize=7)
    lim = [0, 1.6]; ax.plot(lim, lim, "k--", lw=0.8)
ax.set(xlabel="NeuralFoil cl", ylabel="CFD cl (slab → section)", title="CFD against NeuralFoil" + (" — MOCK (not physics)" if DEBUG else ""))

**What confirms the choice.** The CFD should keep the same order at α 4° (cruise) and at α 10° (towards the stall);
its absolute cl reads low on a quick mesh and a short slab (the lifting-line correction is approximate). If the order
flips, run `QUALITY = "screening"` and more angles before changing the wing.